# Evaluation 2:
Meal Plan Recommender Evaluation: Evaluates the simple RAG and the agents.

- For how to setup the vector store, refer to the `vector_store_food_RAG.ipynb` notebook.
- For how to setup the graph store, refer to the `init_graph_store_1.ipynb` and the `init_graph_store_2.ipynb` notebooks.
- On how to generate the question-answer pairs, refer to the `qa_generation.ipynb` notebook.

## 1. Setup
### 1.1 Data

In [ ]:
import nest_asyncio
from llama_index.core import Settings
from llama_index.core.evaluation import CorrectnessEvaluator
from llama_index.core.evaluation import FaithfulnessEvaluator
from llama_index.core.evaluation import GuidelineEvaluator
from llama_index.core.evaluation import RelevancyEvaluator
from llama_index.core.evaluation.guideline import EvaluationData
from llama_index.embeddings.litellm import LiteLLMEmbedding
from llama_index.llms.litellm import LiteLLM
from pathlib import Path

from src.food_rag.agents import workflow_runner
from src.food_rag.evaluation import eval_parser
from src.food_rag.evaluation import retriever_evaluator
from src.food_rag.evaluation.eval_parser import CorrectingPydanticOutputParser
from src.food_rag.evaluation.retriever_evaluator import RUN_PATH
from src.food_rag.query_engines.rag import RecipeRAGQueryEngine
from src.food_rag.retriever import pandas_retriever
from src.food_rag.tools import simple_recommender_tools
from src.food_rag.tools import workflow_tools
from src.food_rag.utils import embedding_helper, hummus_docify
from src.food_rag.utils import llm_helper
from src.food_rag.utils import qa_helper
from src.food_rag.utils.embedding_helper import QA_SUB_PATH
from src.persona import qa_generator
from src.utils import load_util
from src.food_rag.evaluation import llm_evaluator

nest_asyncio.apply()  # Required to run the notebook in Jupyter for disabling async operations in the notebook

# log litellm
#litellm._turn_on_debug()

In [ ]:
## Output Paths
data_path = './data'
embedding_path = data_path + '/embeddings/snowflake_l'
qa_name = 'small_1'
run_name = 'eval_2'

# Load configs
nutrition_config = load_util.get_config('config/nutrition_config.yml')
persona_config = load_util.get_config('config/persona_config.yml')

# Load key
api_key = load_util.load_file('config/api_key')

In [ ]:
# Set eval params
qa_set_size = 10  # TODO define for given use-case
correctness_score_threshold = 3.0  # The threshold for the correctness evaluator between 0 and 5 where 5 is the best score
guidelines = "The response must recommend a meal plan. A meal plan consists of one recommended recipe for each meal type (e.g. breakfast, lunch, dinner). The meal plan must avoid disliked things or allergies, and can optionally include liked things. It should be understandable why the meal plan was recommended."  # The guidelines applied to the guideline evaluator

In [ ]:
# Init LLMs
llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/llama3.1",
    max_retries=100,
    cooldown_time=60,
    reuse_client=False,
)

eval_llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/deepseekr1",
    max_retries=100,
    cooldown_time=60,
    reuse_client=False,
)

eval_llm = llm  # TODO REMOVE THIS LINE, ONLY FOR DEBUGGING

# MUST BE THE SAME EMBED MODEL AS THE ONE USED FOR IMPORT --> check variable embedding_path
embed_model = LiteLLMEmbedding(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model_name="openai/Snowflake/snowflake-arctic-embed-l",
)

# Set Global Settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

In [ ]:
# Load the current recipes data
hummus_data = load_util.get_current_hummus(data_path)
recipes_df = hummus_data['recipes_df']

# Clean df
cleaned_recipes_df = recipes_df.copy()
cleaned_recipes_df = hummus_docify.stringify_nested_lists(cleaned_recipes_df)
cleaned_recipes_df = hummus_docify.stringify_nested_dicts(cleaned_recipes_df)
cleaned_recipes_df = cleaned_recipes_df.rename(
    columns={'duration': 'duration [min]', 'ingredients': 'ingredient_quantities',
             'ingredient_names': 'ingredients', 'direction_size': 'number_of_directions',
             'ingredients_sizes': 'number_of_ingredients'})
cleaned_recipes_df = cleaned_recipes_df.drop(
    columns=['last_changed_date', 'food_kg_locator', 'recipe_url', 'normalization_comment'])

# Docify the recipes and add their embeddings
recipe_loaded_doc_dict, _ = embedding_helper.load_existing_batches(embedding_path + '/recipe_documents')
cleaned_recipe_docs = embedding_helper.extend_doc_embeddings(
    hummus_docify.recipes_to_docs(cleaned_recipes_df, verbose=True), recipe_loaded_doc_dict)

In [ ]:
# Load the embeddings
ingredient_embeddings, _ = embedding_helper.load_existing_batches(embedding_path + '/ingredient_documents')
food_product_embeddings, _ = embedding_helper.load_existing_batches(embedding_path + '/food_product_documents',
                                                                    nodes_only=True,
                                                                    max_batches=50)  # TODO MAX BATCHES ONLY FOR DEBUG
food_on_embeddings, _ = embedding_helper.load_existing_batches(embedding_path + '/food_on_documents', nodes_only=True)

In [ ]:
# Load question-answer pairs & their embeddings (only the recipe qa's)
question_answer_sets = qa_generator.load_qa_set(data_path, sub_dir_name=qa_name, recipes_df=recipes_df)
qa_recipe_doc_dict, _ = embedding_helper.load_existing_batches(
    embedding_path + QA_SUB_PATH + qa_name + '/qa_recipe_documents')

# Clean the qa set and add the embeddings
qa_set = qa_helper.get_best_questions(question_answer_sets['recipe_question_answer_set'], limit_size=qa_set_size,
                                      discard_empty_qas=True)
qa_set = qa_helper.extend_qa_embeddings(qa_set, qa_recipe_doc_dict)
qa_set = qa_helper.get_recipes_from_ids(qa_set, cleaned_recipes_df)
qa_set = qa_helper.get_contexts_qa_set(qa_set)

# Set questions and contexts (valid recipes) for the evaluation
questions = [qa['question'] for qa in qa_set.values()]
# valid_recipes = [qa['answer'] for qa in qa_set.values()] # valid recipe infos --> very time consuming as faithfulness and relevancy evaluators create a SummaryIndex over all recipes --> faster (but not as precise) to have just e.g. all valid recipe titles in one string
valid_recipe_titles = [qa['recipe_titles'] for qa in qa_set.values()]  # valid recipe titles

# Set the reference texts for the correctness evaluation
references_recipe =  ["Recommendation of a recipe. A recipe includes at least a title, nutrition information, a description, and directions for preparation/cooking."] * len(questions)
references_meal_plan = ["Recommendation of a meal plan. A meal plan includes at least one meal type, and one recipe for each present meal type (breakfast, lunch, dinner, snack). Each recipe includes at least a title, nutrition information, a description, and directions for preparation/cooking."] * len(questions)

### 1.2 Retriever

In [ ]:
# Set up the retriever
retriever = pandas_retriever.PandasQueryRetriever(df=cleaned_recipes_df, verbose=False)

## 2. Evaluation
### 2.1 Simple recommender

In [ ]:
# Setup the evaluation
# We do not use guideline as no meal plans are returned here!
runner = eval_parser.ContextBatchEvalRunner(
    save_path=data_path + "/runs/" + run_name,
    file_name="eval_runner_simple_responses_temp.json",
    evaluators={
        "faithfulness": llm_evaluator.SyncFaithfulnessEvaluator(llm=eval_llm, eval_template=eval_parser.FAITHFULNESS_EVAL_TEMPLATE),
        "relevancy": llm_evaluator.SyncRelevancyEvaluator(llm=eval_llm, eval_template=eval_parser.RELEVANCY_EVAL_TEMPLATE),
        "correctness": llm_evaluator.SyncCorrectnessEvaluator(llm=eval_llm, parser_function=eval_parser.score_reasoning_parser,
                                                              eval_template=eval_parser.CORRECTNESS_EVAL_TEMPLATE,
                                                              score_threshold=correctness_score_threshold),
    },
    show_progress=True,
)

#### 2.1.1 Recipe Queries + tfidf Recommender

In [ ]:
# Pretrain simple recommender/similarities
recommender = simple_recommender_tools.TfidfRecommender()
recommender.train(cleaned_recipe_docs)

In [ ]:
# Run Recommender on qa-results with top_k=1
best_matches = recommender.recommend_top1_batch(questions, cleaned_recipes_df)
responses_1 = simple_recommender_tools.matches_to_response(best_matches)

In [ ]:
# evaluate the questions
eval_results_simple = runner.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=responses_1,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_1 = llm_helper.serialize_and_calc_evaluation_result(eval_results_simple)

# Print the results
llm_helper.print_serialized_scores(serialized_results_1)

In [ ]:
# Save the results
load_util.store_file(serialized_results_1, data_path + "/runs/" + run_name + "/tfidf_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_1 = load_util.load_file(data_path + "/runs/" + run_name + "/tfidf_eval_results.pkl")

#### 2.1.2 Recipe Queries + keyword overlap Recommender

In [ ]:
# Pretrain simple recommender/similarities
recommender = simple_recommender_tools.KeywordOverlapRecommender()
recommender.train(cleaned_recipe_docs)

In [ ]:
# Run Recommender on qa-results with top_k=1
best_matches = recommender.recommend_top1_batch(questions, cleaned_recipes_df)
responses_2 = simple_recommender_tools.matches_to_response(best_matches)

In [ ]:
# evaluate the questions
eval_results_simple = runner.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=responses_2,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_2 = llm_helper.serialize_and_calc_evaluation_result(eval_results_simple)

# Print the results
llm_helper.print_serialized_scores(serialized_results_2)

In [ ]:
# Save the results
load_util.store_file(serialized_results_2, data_path + "/runs/" + run_name + "/keyword_overlap_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_2 = load_util.load_file(data_path + "/runs/" + run_name + "/keyword_overlap_eval_results.pkl")

#### 2.1.3 Recipe Queries + jaccard Recommender


In [ ]:
# Pretrain simple recommender/similarities
recommender = simple_recommender_tools.JaccardRecommender()
recommender.train(cleaned_recipe_docs)

In [ ]:
# Run Recommender on qa-results with top_k=1
best_matches = recommender.recommend_top1_batch(questions, cleaned_recipes_df)
responses_3 = simple_recommender_tools.matches_to_response(best_matches)

In [ ]:
# evaluate the questions
eval_results_simple = runner.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=responses_3,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_3 = llm_helper.serialize_and_calc_evaluation_result(eval_results_simple)

# Print the results
llm_helper.print_serialized_scores(serialized_results_3)

In [ ]:
# Save the results
load_util.store_file(serialized_results_3, data_path + "/runs/" + run_name + "/jaccard_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_3 = load_util.load_file(data_path + "/runs/" + run_name + "/jaccard_eval_results.pkl")

#### 2.1.4 Recipe Queries + bigram overlap Recommender


In [ ]:
# Pretrain simple recommender/similarities
recommender = simple_recommender_tools.BigramOverlapRecommender()
recommender.train(cleaned_recipe_docs)

In [ ]:
# Run Recommender on qa-results with top_k=1
best_matches = recommender.recommend_top1_batch(questions, cleaned_recipes_df)
responses_4 = simple_recommender_tools.matches_to_response(best_matches)

In [ ]:
# evaluate the questions
eval_results_simple = runner.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=responses_4,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_4 = llm_helper.serialize_and_calc_evaluation_result(eval_results_simple)

# Print the results
llm_helper.print_serialized_scores(serialized_results_4)

In [ ]:
# Save the results
load_util.store_file(serialized_results_4, data_path + "/runs/" + run_name + "/bigram_overlap_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_4 = load_util.load_file(data_path + "/runs/" + run_name + "/bigram_overlap_eval_results.pkl")

#### 2.1.5 Recipe Queries + bow cosine Recommender


In [ ]:
# Pretrain simple recommender/similarities
recommender = simple_recommender_tools.BoWCosineRecommender()
recommender.train(cleaned_recipe_docs)

In [ ]:
# Run Recommender on qa-results with top_k=1
best_matches = recommender.recommend_top1_batch(questions, cleaned_recipes_df)
responses_5 = simple_recommender_tools.matches_to_response(best_matches)

In [ ]:
# evaluate the questions
eval_results_simple = runner.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=responses_5,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_5 = llm_helper.serialize_and_calc_evaluation_result(eval_results_simple)

# Print the results
llm_helper.print_serialized_scores(serialized_results_5)

In [ ]:
# Save the results
load_util.store_file(serialized_results_5, data_path + "/runs/" + run_name + "/bow_cosine_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_5 = load_util.load_file(data_path + "/runs/" + run_name + "/bow_cosine_eval_results.pkl")

#### 2.1.6 Recipe Queries + doc2vec Recommender


In [ ]:
# Pretrain simple recommender/similarities
recommender = simple_recommender_tools.Doc2VecRecommender()
recommender.train(cleaned_recipe_docs)

In [ ]:
# Run Recommender on qa-results with top_k=1
best_matches = recommender.recommend_top1_batch(questions, cleaned_recipes_df)
responses_6 = simple_recommender_tools.matches_to_response(best_matches)

In [ ]:
# evaluate the questions
eval_results_simple = runner.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=responses_6,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_6 = llm_helper.serialize_and_calc_evaluation_result(eval_results_simple)

# Print the results
llm_helper.print_serialized_scores(serialized_results_6)

In [ ]:
# Save the results
load_util.store_file(serialized_results_6, data_path + "/runs/" + run_name + "/doc2vec_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_6 = load_util.load_file(data_path + "/runs/" + run_name + "/doc2vec_eval_results.pkl")

#### 2.1.7 Recipe Queries + embedding similarity Recommender


In [ ]:
# Pretrain simple recommender/similarities
recommender = simple_recommender_tools.PrecomputedEmbeddingRecommender()
recommender.train(cleaned_recipe_docs)

In [ ]:
# Run Recommender on qa-results with top_k=1
best_matches = recommender.recommend_top1_batch(questions, cleaned_recipes_df)
responses_7 = simple_recommender_tools.matches_to_response(best_matches)

In [ ]:
# evaluate the questions
eval_results_simple = runner.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=responses_7,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_7 = llm_helper.serialize_and_calc_evaluation_result(eval_results_simple)

# Print the results
llm_helper.print_serialized_scores(serialized_results_7)

In [ ]:
# Save the results
load_util.store_file(serialized_results_7,
                     data_path + "/runs/" + run_name + "/embedding_similarity_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_7 = load_util.load_file(data_path + "/runs/" + run_name + "/embedding_similarity_eval_results.pkl")

#### 2.1.8 Recipe Queries + BM25

In [ ]:
# Pretrain simple recommender/similarities
recommender = simple_recommender_tools.BM25Recommender()
recommender.train(cleaned_recipe_docs)

In [ ]:
# Run Recommender on qa-results with top_k=1
best_matches = recommender.recommend_top1_batch(questions, cleaned_recipes_df)
responses_0 = simple_recommender_tools.matches_to_response(best_matches)

In [ ]:
# evaluate the questions
eval_results_simple = runner.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=responses_0,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_0 = llm_helper.serialize_and_calc_evaluation_result(eval_results_simple)
# Print the results
llm_helper.print_serialized_scores(serialized_results_0)

In [ ]:
# Save the results
load_util.store_file(serialized_results_0,
                     data_path + "/runs/" + run_name + "/bm25_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_0 = load_util.load_file(data_path + "/runs/" + run_name + "/bm25_eval_results.pkl")

### 2.2 Recipe Recommendation RAG

In [ ]:
# Set up the RAG query engine
query_engine_rag = RecipeRAGQueryEngine(retriever=retriever, llm=llm, ranking_strategy='rating', k=1, llm_calls=2,
                                        return_full_recipes=True, return_meal_plan=True, debug=True)
# NOTE: For the pandas-to-text set return_meal_plan=False, then it does not use the LLM

In [ ]:
# # answer & evaluate the questions
# runner = eval_parser.ContextBatchEvalRunner(
#     save_path=data_path + "/runs/" + run_name,
#     file_name="eval_runner_rag_responses_temp.pkl",
#     evaluators={
#         "faithfulness": FaithfulnessEvaluator(llm=eval_llm, eval_template=eval_parser.FAITHFULNESS_EVAL_TEMPLATE),
#         "relevancy": RelevancyEvaluator(llm=eval_llm, eval_template=eval_parser.RELEVANCY_EVAL_TEMPLATE),
#         "guideline": GuidelineEvaluator(llm=eval_llm, guidelines=guidelines,
#                                         eval_template=eval_parser.GUIDELINE_EVAL_TEMPLATE,
#                                         output_parser=CorrectingPydanticOutputParser(
#                                             output_default=EvaluationData(feedback="", passing=False),
#                                             output_cls=EvaluationData
#                                         )),
#         "correctness": CorrectnessEvaluator(llm=eval_llm, parser_function=eval_parser.score_reasoning_parser,
#                                             eval_template=eval_parser.CORRECTNESS_EVAL_TEMPLATE,
#                                             score_threshold=correctness_score_threshold),
#     },
#     show_progress=True,
# )
#
# eval_results_RAG = runner.evaluate_queries(  # of type Dict[str, List[EvaluationResult]] where keys are metric names
#     # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_queries(....
#     query_engine=query_engine_rag,
#     queries=questions,
#     contexts=valid_recipe_titles,
#     reference=references_recipe,
# )
# # Get responses for later storage from one of the metric results, doesn't matter which one
# rag_responses = [eval_result.response for eval_result in eval_results_RAG['faithfulness']]

In [ ]:
tool_rag = workflow_tools.MealPlanRAGTool(llm=llm, recipe_query_engine=query_engine_rag, verbose=False)

# Run the workflow for each question (and also loads previously saved results)
question_and_responses_rag = workflow_runner.run_all_questions_on_tool(
    questions=questions,
    tool=tool_rag,
    data_path=data_path,
    sub_dir_name=run_name,
    file_name="rag_responses_temp.pkl",
)

# Make sure the order of the responses is the same as the questions
rag_responses = workflow_runner.to_ordered_response(questions=questions,
                                                    question_and_responses=question_and_responses_rag)

In [ ]:
# answer & evaluate the questions
runner_rag = eval_parser.ContextBatchEvalRunner(
    save_path=data_path + "/runs/" + run_name,
    file_name="eval_runner_rag_responses_temp.json",
    evaluators={
        "faithfulness": llm_evaluator.SyncFaithfulnessEvaluator(llm=eval_llm, eval_template=eval_parser.FAITHFULNESS_EVAL_TEMPLATE),
        "relevancy": llm_evaluator.SyncRelevancyEvaluator(llm=eval_llm, eval_template=eval_parser.RELEVANCY_EVAL_TEMPLATE),
        "guideline": llm_evaluator.SyncGuidelineEvaluator(llm=eval_llm, guidelines=guidelines,
                                                          eval_template=eval_parser.GUIDELINE_EVAL_TEMPLATE,
                                                          output_parser=CorrectingPydanticOutputParser(
                                                              output_default=EvaluationData(feedback="", passing=False),
                                                              output_cls=EvaluationData
                                                          )),
        "correctness": llm_evaluator.SyncCorrectnessEvaluator(llm=eval_llm, parser_function=eval_parser.score_reasoning_parser,
                                                              eval_template=eval_parser.CORRECTNESS_EVAL_TEMPLATE,
                                                              score_threshold=correctness_score_threshold),
    },
    show_progress=True,
)

eval_results_RAG = runner_rag.evaluate_responses(  # of type Dict[str, List[EvaluationResult]] where keys are metric names
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_queries(....
    queries=questions,
    responses=rag_responses,
    contexts=valid_recipe_titles,
    reference=references_meal_plan,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_8 = llm_helper.serialize_and_calc_evaluation_result(eval_results_RAG)

# Print the results
llm_helper.print_serialized_scores(serialized_results_8)

In [ ]:
# Save the results
load_util.store_file(serialized_results_8, data_path + "/runs/" + run_name + "/rag_eval_results.pkl")

In [ ]:
# Load the results
# serialized_results_8 = load_util.load_file(data_path + "/runs/" + run_name + "/rag_eval_results.pkl")

### 2.3 Heuristic Agents

In [ ]:
# Set up the Heuristic query engine
query_engine_heuristic = RecipeRAGQueryEngine(retriever=retriever, llm=llm, ranking_strategy='rating', k=1, llm_calls=2,
                                              return_full_recipes=False, return_meal_plan=False, debug=True)


#### 2.3.1 Heuristic Agent #1

In [ ]:
tool_1 = workflow_tools.MealPlanWorkflowTool(hummus_data=hummus_data, recipe_df=cleaned_recipes_df, llm=llm,
                                             recipe_query_engine=query_engine_heuristic, verbose=False)

# Run the workflow for each question (and also loads previously saved results)
question_and_responses_1 = workflow_runner.run_all_questions_on_tool(
    questions=questions,
    tool=tool_1,
    data_path=data_path,
    sub_dir_name=run_name,
    file_name="heuristic_responses_temp_1.pkl",
    objectives=[]
)

# Make sure the order of the responses is the same as the questions
heuristic_responses_1 = workflow_runner.to_ordered_response(questions=questions,
                                                            question_and_responses=question_and_responses_1)

In [ ]:
# answer & evaluate the questions
runner_1 = eval_parser.ContextBatchEvalRunner(
    save_path=data_path + "/runs/" + run_name,
    file_name="eval_runner_heuristic_1_responses_temp.json",
    evaluators={
        "faithfulness": FaithfulnessEvaluator(llm=eval_llm, eval_template=eval_parser.FAITHFULNESS_EVAL_TEMPLATE),
        "relevancy": RelevancyEvaluator(llm=eval_llm, eval_template=eval_parser.RELEVANCY_EVAL_TEMPLATE),
        "guideline": GuidelineEvaluator(llm=eval_llm, guidelines=guidelines,
                                        eval_template=eval_parser.GUIDELINE_EVAL_TEMPLATE,
                                        output_parser=CorrectingPydanticOutputParser(
                                            output_default=EvaluationData(feedback="", passing=False),
                                            output_cls=EvaluationData
                                        )),
        "correctness": CorrectnessEvaluator(llm=eval_llm, parser_function=eval_parser.score_reasoning_parser,
                                            eval_template=eval_parser.CORRECTNESS_EVAL_TEMPLATE,
                                            score_threshold=correctness_score_threshold),
    },
    show_progress=True,
)

eval_results_heuristic_agent_1 = runner_1.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=heuristic_responses_1,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_9 = llm_helper.serialize_and_calc_evaluation_result(eval_results_heuristic_agent_1)

# Print the results
llm_helper.print_serialized_scores(serialized_results_9)

In [ ]:
# Save the results
load_util.store_file(serialized_results_9, data_path + "/runs/" + run_name + "/heuristic_agent_eval_results_1.pkl")

In [ ]:
# Load the results
# serialized_results_9 = load_util.load_file(data_path + "/runs/" + run_name + "/heuristic_agent_eval_results_1.pkl")

#### 2.3.1 Heuristic Agent #2 (short version)

In [ ]:
tool_2 = workflow_tools.MealPlanShortWorkflowTool(recipe_df=cleaned_recipes_df, llm=llm,
                                                  recipe_query_engine=query_engine_heuristic, verbose=False)

# Run the workflow for each question (and also loads previously saved results)
question_and_responses_2 = workflow_runner.run_all_questions_on_tool(
    questions=questions,
    tool=tool_2,
    data_path=data_path,
    sub_dir_name=run_name,
    file_name="heuristic_responses_temp_2.pkl",
)

# Make sure the order of the responses is the same as the questions
heuristic_responses_2 = workflow_runner.to_ordered_response(questions=questions,
                                                            question_and_responses=question_and_responses_2)

In [ ]:
# answer & evaluate the questions
runner_2 = eval_parser.ContextBatchEvalRunner(
    save_path=data_path + "/runs/" + run_name,
    file_name="eval_runner_heuristic_2_responses_temp.json",
    evaluators={
        "faithfulness": FaithfulnessEvaluator(llm=eval_llm, eval_template=eval_parser.FAITHFULNESS_EVAL_TEMPLATE),
        "relevancy": RelevancyEvaluator(llm=eval_llm, eval_template=eval_parser.RELEVANCY_EVAL_TEMPLATE),
        "guideline": GuidelineEvaluator(llm=eval_llm, guidelines=guidelines,
                                        eval_template=eval_parser.GUIDELINE_EVAL_TEMPLATE,
                                        output_parser=CorrectingPydanticOutputParser(
                                            output_default=EvaluationData(feedback="", passing=False),
                                            output_cls=EvaluationData
                                        )),
        "correctness": CorrectnessEvaluator(llm=eval_llm, parser_function=eval_parser.score_reasoning_parser,
                                            eval_template=eval_parser.CORRECTNESS_EVAL_TEMPLATE,
                                            score_threshold=correctness_score_threshold),
    },
    show_progress=True,
)

eval_results_heuristic_agent_2 = runner_2.evaluate_responses(
    # in a non-notebook we would now use 'await' and 'aevaluate': await runner.aevaluate_responses(....
    queries=questions,
    responses=heuristic_responses_2,
    contexts=valid_recipe_titles,
    reference=references_recipe,
)

In [ ]:
# Serialize the results and calculate the overall scores
serialized_results_10 = llm_helper.serialize_and_calc_evaluation_result(eval_results_heuristic_agent_2)

# Print the results
llm_helper.print_serialized_scores(serialized_results_10)

In [ ]:
# Save the results
load_util.store_file(serialized_results_10, data_path + "/runs/" + run_name + "/heuristic_agent_eval_results_2.pkl")

In [ ]:
# Load the results
# serialized_results_10 = load_util.load_file(data_path + "/runs/" + run_name + "/heuristic_agent_eval_results_2.pkl")

# 3. Summary

In [ ]:
# Summarize the evaluation results in a df
results_df = retriever_evaluator.summarize_results(
    {
        'Recipe Queries + tfidf Recommender (1)': serialized_results_1,
        'Recipe Queries + keyword overlap Recommender (2)': serialized_results_2,
        'Recipe Queries + jaccard Recommender (3)': serialized_results_3,
        'Recipe Queries + bigram overlap Recommender (4)': serialized_results_4,
        'Recipe Queries + bow cosine Recommender (5)': serialized_results_5,
        'Recipe Queries + doc2vec Recommender (6)': serialized_results_6,
        'Recipe Queries + embedding similarity Recommender (7)': serialized_results_7,
        'Recipe Recommendation RAG (8)': serialized_results_8,
        'Heuristic Agent #1 (9)': serialized_results_9,
        'Heuristic Agent #2 (10)': serialized_results_10,
    }, key='overall_score'
)
results_df

In [ ]:
# Store the evaluation results
load_util.store_file(data=results_df, path=Path(data_path, RUN_PATH, run_name, 'evaluation_results'))